<!--
Copyright (c) 2026 F. Hoffmann-La Roche Ltd
Authors:
- ERP Technology Platform Team <glospot_members@msxdl.roche.com>
-->

# Read and cache function module metadata

Before it calls a function module, `dragonrfc` needs its *metadata*: the names,
ABAP types and lengths of its parameters, and the layout of every structure and
table behind them. The SAP NW RFC SDK reads this from the DDIC of the target
system on first use and keeps it in a cache, the *repository*. You can read that
metadata yourself, which is useful to discover a signature, to check it against
what your code assumes, and to warm up or persist the cache.

This notebook uses `SSFR_PSE_CHECK` (function group `SSFR`) on system **EH4**, because
its signature has a structure (`IS_STRUST_IDENTITY`) and a table (`ET_BAPIRET2`),
and shows how to:

1. read the signature of a function module (`describe_function`);
2. read the layout of a structure or table line type (`describe_type`);
3. turn metadata into plain data (`dataclasses.asdict`);
4. save the cache to a JSON file and load it again (`Repository`);
5. fill the cache for many functions and types in one roundtrip (`prefetch`);
6. do all of the above with the asyncio client.

The metadata comes back as immutable dataclasses (`FunctionMetadata`,
`ParameterMetadata`, `ExceptionMetadata`, `TypeMetadata`, `FieldMetadata`) whose
fields map one to one to the SDK descriptors. The logon uses **SNC with
Kerberos**, so no password appears anywhere.

## Prerequisites

1. The SAP NW RFC SDK is installed and `dragonrfc` is built (`task setup`).
2. MIT Kerberos 5 is installed. Its GSSAPI library is the SNC library:
   `libgssapi_krb5.so.2` on Linux, `libgssapi_krb5.dylib` on macOS and
   `gssapi64.dll` on Windows.
3. You hold a valid Kerberos ticket for your user. Get one with
   `kinit prietoj4@EMEA.ROCHE.COM` and check it with `klist`.
4. Your user in EH4, client 010, is mapped to the SNC name
   `p:prietoj4@EMEA.ROCHE.COM` (transaction `SU01`, tab *SNC*) and may read the
   repository over RFC (`S_RFC`; the SDK uses `RFC_GET_FUNCTION_INTERFACE`).

## Step 1: Logon parameters

`RFCClient` takes the SDK logon parameters as keyword arguments, all of them
`str`. All examples below share the same dictionary.

**Load-balanced logon through the message server**

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `mshost` | `eh4scs.sap.roche.com` | Host of the message server (ASCS instance). |
| `msserv` | `3600` | Message server port (`36<instance number>`). |
| `sysid` | `EH4` | System ID; the message server serves only this system. |
| `group` | `SPACE` | Logon group (`SMLG`); the message server picks the least loaded application server in it. |
| `client` | `010` | SAP client. |

**SNC instead of user and password**

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `snc_mode` | `1` | Turns SNC on. The SAP user is derived from the SNC name, so `user` and `passwd` are not set. |
| `snc_qop` | `9` | Quality of protection: `9` asks for the maximum the SNC library offers (integrity and privacy, i.e. encryption). |
| `snc_myname` | `p:prietoj4@EMEA.ROCHE.COM` | Your own SNC name: the Kerberos principal of your ticket. |
| `snc_partnername` | `p:sapEH4/EH4@EMEA.ROCHE.COM` | SNC name of the SAP system: the Kerberos service principal of EH4. |
| `snc_lib` | GSSAPI library | Path or name of the MIT Kerberos GSSAPI library. |

The cell finds the GSSAPI library with `ctypes.util.find_library`, which works on
every OS when MIT Kerberos is installed in the default location. If it does not
find it (for example, Homebrew's keg-only `krb5` on macOS), set the `SNC_LIB`
environment variable to the full path of the library before starting Jupyter.

In [2]:
import ctypes.util
import os

import dragonrfc

SNC_LIB = os.environ.get("SNC_LIB") or ctypes.util.find_library(
    "gssapi64" if os.name == "nt" else "gssapi_krb5",
)
assert SNC_LIB, "MIT Kerberos GSSAPI library not found: set SNC_LIB"

LOGON = {
    # Load balancing through the message server
    "mshost": "eh4scs.sap.roche.com",
    "msserv": "3600",
    "sysid": "EH4",
    "group": "SPACE",
    "client": "010",
    # SNC with Kerberos (GSSAPI)
    "snc_mode": "1",
    "snc_qop": "9",
    "snc_myname": "p:prietoj4@EMEA.ROCHE.COM",
    "snc_partnername": "p:sapEH4/EH4@EMEA.ROCHE.COM",
    "snc_lib": SNC_LIB,
}

print(f"dragonrfc {dragonrfc.__version__}, SDK {dragonrfc.sdk_version()}")
print(f"SNC library: {SNC_LIB}")

ModuleNotFoundError: No module named 'dragonrfc._core'

## Step 2: Check connectivity

Before calling any function module, make sure EH4 is reachable and accepts the
SNC logon. Opening the connection already performs the whole logon (message
server lookup, SNC handshake with your Kerberos ticket, user check);
`ping` then sends a round trip over the open connection.

Failures surface as typed exceptions, which the cell reports instead of a
traceback:

- `dragonrfc.RFCCommunicationError`: the message server or the application
  server cannot be reached (wrong host or port, firewall, VPN off), or the SNC
  library failed (for example, no valid Kerberos ticket).
- `dragonrfc.RFCLogonError`: EH4 rejected the logon (SNC name not mapped to a
  user, user locked, wrong client).

The same check is shown with both clients. Expected output: `EH4 reachable (sync)`
and `EH4 reachable (async)`. Continue only if both succeed.

In [ ]:
try:
    with dragonrfc.RFCClient(**LOGON) as conn:
        conn.ping()
except (dragonrfc.RFCCommunicationError, dragonrfc.RFCLogonError) as error:
    print(f"EH4 not reachable (sync): {type(error).__name__}: {error.message}")
else:
    print("EH4 reachable (sync)")

try:
    async with await dragonrfc.AsyncRFCClient.open(**LOGON) as aconn:
        await aconn.ping()
except (dragonrfc.RFCCommunicationError, dragonrfc.RFCLogonError) as error:
    print(f"EH4 not reachable (async): {type(error).__name__}: {error.message}")
else:
    print("EH4 reachable (async)")

## Step 3: Read the signature of a function module (synchronous)

`describe_function` returns a `FunctionMetadata` with:

- `name`: the function module name;
- `parameters`: a tuple of `ParameterMetadata`, in declaration order;
- `exceptions`: a tuple of `ExceptionMetadata`, the classic exceptions the
  function module can raise (`key` and `message`).

Each parameter has a `name`, an ABAP `type` (`ABAPType`; a value the enum does
not know stays a plain `int`), a `direction` (`Direction.IMPORT`, `EXPORT`,
`CHANGING` or `TABLES`), its length in bytes in a Unicode system (`uc_length`),
whether it is `optional`, and, for structures and tables, the `structure` type
behind it. `default` and `text` are the default value and description from
transaction `SE37`; they are blank when the system does not maintain them.

The first call reads the DDIC over the connection; later calls for the same
function module are answered from the cache. A function module that does not
exist raises `RFCError`.

Note that `Direction.CHANGING` is `IMPORT | EXPORT` and `Direction.TABLES`
includes both, so to test for "can be passed in" use `p.direction & Direction.IMPORT`,
and to name a direction use `p.direction.name`.

In [ ]:
with dragonrfc.RFCClient(**LOGON) as conn:
    meta = conn.describe_function("SSFR_PSE_CHECK")

print(meta.name)
for p in meta.parameters:
    abap_type = getattr(p.type, "name", p.type)
    structure = p.structure.name if p.structure else "-"
    print(
        f"  {p.direction.name:<7} {p.name:<20} {abap_type:<10} "
        f"len={p.uc_length:<5} optional={p.optional!s:<5} structure={structure}",
    )
print("exceptions:", [e.key for e in meta.exceptions] or "none")

## Step 4: Read the layout of a structure or table type (synchronous)

A structure or table parameter carries its type in `structure`: a `TypeMetadata`
with `name` (the DDIC type, e.g. `SSF_S_STRUST_IDENTITY`), the total length in
bytes in a non-Unicode and a Unicode system (`nuc_length`, `uc_length`) and
`fields`, a tuple of `FieldMetadata`. For a table parameter, `structure` is the
*line* type.

Each field has a `name`, an ABAP `type`, `uc_length` and `uc_offset` (and the
`nuc_*` counterparts), `decimals` for packed numbers and, when it is itself a
structure or table, a nested `structure`. The `walk` helper below follows that
nesting.

`describe_type` reads a type directly by its DDIC name, without going through a
function module. For a parameter's type it returns the same description
(dataclasses compare by value), which the last line checks.

In [ ]:
def walk(type_meta, indent=0):
    """Print a type and, indented, the types nested in its fields."""
    pad = "  " * indent
    print(f"{pad}{type_meta.name} (uc_length={type_meta.uc_length})")
    for f in type_meta.fields:
        abap_type = getattr(f.type, "name", f.type)
        print(
            f"{pad}  {f.name:<14} {abap_type:<10} "
            f"offset={f.uc_offset:<4} length={f.uc_length}",
        )
        if f.structure:
            walk(f.structure, indent + 2)


params = {p.name: p for p in meta.parameters}
with dragonrfc.RFCClient(**LOGON) as conn:
    ddic = conn.describe_type(params["IS_STRUST_IDENTITY"].structure.name)

walk(ddic)
print()
walk(params["ET_BAPIRET2"].structure)
print()
print("same as the parameter's type:", ddic == params["IS_STRUST_IDENTITY"].structure)

## Step 5: Turn metadata into plain data

The metadata classes are frozen dataclasses, so `dataclasses.asdict` converts
them, with every nested type, into plain dictionaries and lists. That is what
you want to log, compare, store as JSON or feed to another tool. `ABAPType` and
`Direction` are integer enums, so `json` writes them as numbers.

The cell below turns the `IS_STRUST_IDENTITY` type into JSON and prints its
first field.

In [ ]:
import dataclasses
import json

as_dict = dataclasses.asdict(ddic)
print(json.dumps(as_dict["fields"][0], indent=2))

## Step 6: Save and load the repository cache

The SDK keeps one process-wide cache per system ID. `conn.repository` is a
`Repository` for the system of a connection, and `dragonrfc.Repository("EH4")`
selects the same cache without a connection. It offers:

- `describe_function` / `describe_type`: read from the cache only, never from
  the network. A name that is not cached raises `RFCError`.
- `save(path)`: writes the cached *function* descriptions to a JSON file. The
  format is shared with SAP JCo and SAP NCo.
- `load(path)`: **replaces** the cache with the descriptions in such a file, so
  a program can start without DDIC lookups.
- `evict_function` / `evict_type`: drop one description, for example after the
  signature changed in the backend, so the next call reads it again.
  `clear()` drops everything.

Take care with `load`: the file must match the DDIC of the system. Outdated
metadata can truncate or corrupt data in later calls, so reload the cache from
the system whenever the function modules or their types change. The file keeps
type layouts but not parameter defaults or texts, and after `load` a structure
or table parameter reports the size of a reference (`uc_length` of 8) instead of
its own length, so read the length from `p.structure.uc_length`.

The cell saves the cache to a temporary file, evicts the function to show that
the repository cannot answer without it, loads the file back and reads the
function again, all without a connection.

In [ ]:
import tempfile
from pathlib import Path

with dragonrfc.RFCClient(**LOGON) as conn:
    conn.describe_function("SSFR_PSE_CHECK")  # make sure it is cached
    repo = conn.repository

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "eh4-metadata.json"
    repo.save(path)
    print(f"saved {path.stat().st_size} bytes for system {repo.sysid}")

    repo.evict_function("SSFR_PSE_CHECK")
    try:
        repo.describe_function("SSFR_PSE_CHECK")
    except dragonrfc.RFCError as exc:
        print("after evict:", exc.message)

    dragonrfc.Repository("EH4").load(path)
    print("after load:", repo.describe_function("SSFR_PSE_CHECK").name)

## Step 7: Prefetch many descriptions in one roundtrip (synchronous)

Without help, every function module and type is a separate DDIC lookup the first
time it is used. `prefetch(functions=..., types=...)` fills the cache for a whole
list in one request.

For a single roundtrip, open the connection with the logon parameter
`USE_REPOSITORY_ROUNDTRIP_OPTIMIZATION="1"`. The backend needs the support
package level of SAP Note 1456826; without it the SDK reports
`RFC_METADATA_GET not enabled` for every name.

`prefetch` does not raise for a name that cannot be found. It returns a
dictionary with the error message of every failed lookup, and an empty one when
all succeeded. The cell asks for one existing function, one existing type and one
function that does not exist, so the result shows how a failure looks.

In [ ]:
with dragonrfc.RFCClient(**LOGON, USE_REPOSITORY_ROUNDTRIP_OPTIMIZATION="1") as conn:
    failed = conn.prefetch(
        functions=["SSFR_GET_ALL_STRUST_IDENTITIES", "NO_SUCH_FUNCTION"],
        types=["BAPIRET2"],
    )
    print("failed lookups:", failed)
    cached = conn.repository.describe_function("SSFR_GET_ALL_STRUST_IDENTITIES")
    print("cached:", cached.name, [p.name for p in cached.parameters])

## Step 8: The same with the asyncio client

`AsyncRFCClient` has the same `describe_function`, `describe_type` and `prefetch`
methods, awaited; they run the synchronous method on a worker thread. Calls on
one client run one after the other, so for concurrent lookups use one client per
lookup, as in the earlier examples.

`repository` only touches local state, so it has no async twin: reach it through
`aconn.client`, the underlying `RFCClient`. The cell describes a function and a
type, prefetches, and reads the cache, with the same results as before.

In [ ]:
async with await dragonrfc.AsyncRFCClient.open(
    **LOGON,
    USE_REPOSITORY_ROUNDTRIP_OPTIMIZATION="1",
) as aconn:
    ameta = await aconn.describe_function("SSFR_PSE_CHECK")
    atype = await aconn.describe_type("SSF_S_STRUST_IDENTITY")
    afailed = await aconn.prefetch(types=["BAPIRET2"])
    cached_names = aconn.client.repository.describe_type("BAPIRET2").name

same_layouts = [p.structure for p in ameta.parameters] == [
    p.structure for p in meta.parameters
]
print("same parameter layouts as sync:", same_layouts)
print("same type as sync:", atype == ddic)
print("prefetch failures:", afailed or "none", "| cached:", cached_names)

## Summary

- `describe_function` and `describe_type` return the signature and layouts as
  frozen dataclasses; `dataclasses.asdict` makes them plain data.
- The first lookup reads the DDIC of the system, later ones come from the SDK
  cache, which is shared by all connections to the same system ID.
- `Repository` reads the cache without a connection and saves or loads it as JSON.
  Load only a file that matches the system's current DDIC, and expect blank
  defaults and texts, and 8-byte lengths for structure and table parameters,
  afterwards.
- `prefetch` warms the cache for many functions and types in one roundtrip when the
  connection sets `USE_REPOSITORY_ROUNDTRIP_OPTIMIZATION="1"`; it reports failures
  in its result instead of raising.
- Call `evict_function` or `evict_type` after a signature changed in the backend.
- `AsyncRFCClient` mirrors the network methods; `repository` is reached through
  `aconn.client`.